# PhishGuard AI - 02 Feature Engineering
**Deterministic Extraction of 42 URL Features**

Extracts structural, lexical, statistical, security, keyword, and domain characteristics from URLs.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.config import DEFAULT_RAW_DATASET_CSV, DEFAULT_PROCESSED_DATASET_CSV
from src.data_loader import DatasetLoader
from src.preprocessing import DataPreprocessor
from src.feature_extraction import (
    FEATURE_NAMES,
    extract_features_from_url,
    extract_features_dataframe,
    get_feature_metadata
)

print(f"Total features configured: {len(FEATURE_NAMES)}")

Total features configured: 42


## 1. Single URL Feature Extraction Demonstration

In [2]:
sample_phish = "http://paypal.com.account-verification-service.tk/login.php?session=98214"
sample_legit = "https://github.com/scikit-learn/scikit-learn"

feat_phish = extract_features_from_url(sample_phish)
feat_legit = extract_features_from_url(sample_legit)

comparison_df = pd.DataFrame([feat_legit, feat_phish], index=["Legitimate (GitHub)", "Phishing (PayPal Spoof)"]).T
comparison_df.head(25)

,Legitimate (GitHub),Phishing (PayPal Spoof)
url_length,44.0,73.0
domain_length,10.0,42.0
path_length,26.0,10.0
query_length,0.0,13.0
fragment_length,0.0,0.0
qty_dot,1.0,4.0
qty_hyphen,2.0,2.0
qty_slash,4.0,3.0
qty_questionmark,0.0,1.0
qty_equal,0.0,1.0


## 2. Batch Extraction Across Entire Dataset

In [3]:
loader = DatasetLoader(filepath=DEFAULT_RAW_DATASET_CSV)
df = loader.load()
preprocessor = DataPreprocessor()
df_clean = preprocessor.clean_dataset(df, url_col=loader.url_col, label_col=loader.label_col)

# Extract features
X_features = extract_features_dataframe(df_clean[loader.url_col])
X_features['target_label'] = df_clean['label_normalized'].values

# Save to processed directory
X_features.to_csv(DEFAULT_PROCESSED_DATASET_CSV, index=False)
print(f"Saved processed features to: {DEFAULT_PROCESSED_DATASET_CSV}")
print("Feature Dataframe Shape:", X_features.shape)
X_features.head()

2026-10-02 21:51:31,130 [INFO] DataLoader: Loading dataset from: C:\Users\mubas\OneDrive\Desktop\PB-Hackthon\data\raw\phishing_urls_dataset.csv
2026-10-02 21:51:31,140 [INFO] DataLoader: Dataset loaded successfully with shape: (2500, 2)
2026-10-02 21:51:31,142 [INFO] DataLoader: Detected URL column: 'url', Label column: 'label'
2026-10-02 21:51:31,143 [INFO] Preprocessing: Initial raw rows: 2500
2026-10-02 21:51:31,150 [INFO] Preprocessing: Rows after dropping nulls in critical columns: 2500
2026-10-02 21:51:31,164 [INFO] Preprocessing: Dropped 84 duplicate URL rows. Remaining: 2416
2026-10-02 21:51:31,201 [INFO] Preprocessing: Final cleaned dataset size: 2416
2026-10-02 21:51:31,203 [INFO] Preprocessing: Class Distribution -> Legitimate (0): 1182 (48.9%), Phishing (1): 1234 (51.1%)
2026-10-02 21:51:31,205 [INFO] FeatureExtraction: Extracting 42 features from 2416 URLs...
2026-10-02 21:51:31,254 [INFO] FeatureExtraction: Processed 500/2416 URLs (20.7%)
2026-10-02 21:51:31,312 [INFO] Fe

Saved processed features to: C:\Users\mubas\OneDrive\Desktop\PB-Hackthon\data\processed\extracted_features.csv
Feature Dataframe Shape: (2416, 43)


,url_length,domain_length,path_length,query_length,fragment_length,qty_dot,qty_hyphen,qty_slash,qty_questionmark,qty_equal,...,letter_ratio,url_entropy,max_consecutive_digits,suspicious_pattern_count,is_numeric_heavy_domain,high_subdomain_count,is_suspicious_tld,domain_entropy,domain_vowel_count,target_label
0,74,16,24,25,0,2,0,6,1,3,...,0.6486,4.6660,5,2,0,0,0,3.2028,5,0
1,29,13,8,0,0,3,0,3,0,0,...,0.7586,3.6335,0,2,0,0,0,2.8151,5,0
2,72,21,17,25,0,2,0,4,1,3,...,0.7361,4.6227,5,2,0,0,0,3.4992,5,0
3,59,30,11,10,0,4,1,3,1,1,...,0.7288,4.5244,5,2,0,0,1,3.8981,8,1
4,39,15,16,0,0,2,0,4,0,0,...,0.8205,4.0579,0,2,0,0,0,3.3232,4,0
